# Prediksi Dini Risiko Dropout Mahasiswa

Eksperimen untuk penelitian *Prediksi Dini Risiko Dropout Mahasiswa pada Tiga Tahap Studi Menggunakan Random Forest dengan Analisis Ketergantungan pada Atribut Keuangan*. Notebook ini berdiri sendiri dan dapat dijalankan di Kaggle (utama) atau di shell Nix laptop (cadangan).

**Data.** Realinho, V., Vieira Martins, M., Machado, J., & Baptista, L. (2021). *Predict students' dropout and academic success* [Data set]. UCI Machine Learning Repository. <https://doi.org/10.24432/C5MC89>. Lisensi CC BY 4.0.

**Isi.** Bagian 1 sampai 7 mengikuti tahapan CRISP-DM pada proposal penelitian: pemahaman masalah, pengumpulan data, preprocessing, eksplorasi data, pemodelan, evaluasi, dan pemanfaatan. Pembagian data dilakukan di bagian preprocessing, sebelum eksplorasi, supaya eksplorasi hanya melihat data latih.

**Cara menjalankan.**

- **Kaggle.** Aktifkan internet di pengaturan notebook, lalu jalankan semua sel. Notebook memasang versi pustaka yang dikunci dan mengunduh arsip UCI sendiri. Satu jalan memakan sekitar 35 menit.
- **Laptop.** Jalankan `run-notebook` dari shell Nix repo. Perintah ini menimpa keluaran notebook, `artifacts/`, dan `artifacts.zip`. Skor XGBoost di laptop dapat sedikit berbeda dari Kaggle, jadi artefak final tetap diambil dari jalan di Kaggle.

Artefak ditulis ke `artifacts/` (di Kaggle `/kaggle/working/artifacts/`), lalu dikemas menjadi `artifacts.zip`. Aplikasi Streamlit di repo <https://github.com/MapleKaedee/prediksi-dini-dropout> membaca artefak ini, dan artefak final tersedia di GitHub Release `artifacts-v1` repo itu.

## Persiapan lingkungan

Sel pertama memasang versi pustaka yang dikunci, hanya jika notebook berjalan di Kaggle. Di sana pip dapat menampilkan galat konflik dengan `nilearn` dan saran untuk me-restart kernel. Notebook ini tidak memakai `nilearn`, dan sel kedua menghentikan notebook jika versi scikit-learn tidak sesuai. Jadi kedua pesan itu boleh diabaikan selama sel kedua lolos.

In [1]:
from pathlib import Path

if Path("/kaggle/working").exists():
    %pip install -q scikit-learn==1.9.0 xgboost==3.0.5 shap==0.52.0

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
nilearn 0.14.1 requires scikit-learn!=1.7.0,!=1.9.0,>=1.5.0, but you have scikit-learn 1.9.0 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import hashlib
import io
import itertools
import json
import platform
import time
import urllib.request
import warnings
import zipfile
from datetime import datetime, timezone
from importlib.metadata import version

import joblib
import numpy as np
import pandas as pd
import sklearn

# tqdm memperingatkan jika ipywidgets tidak ada, dan pesannya memuat path lokal.
warnings.filterwarnings("ignore", message="IProgress not found")
import shap
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

# NB-01: pickle scikit-learn terikat pada versinya. Nilai ini harus sama dengan baris %pip di atas.
SKLEARN_PIN = "1.9.0"
assert sklearn.__version__ == SKLEARN_PIN, (
    f"scikit-learn {sklearn.__version__} terpasang, seharusnya {SKLEARN_PIN}. "
    "Di Kaggle: restart kernel, lalu jalankan ulang dari sel pertama. Di laptop: pakai shell Nix."
)

SEED = 42
THRESHOLD = 0.5
MIN_DIFFERENCE = 0.05
START = time.perf_counter()
timings = {"tuning": 0.0, "shap": 0.0}
STATUSES = ["Dropout", "Graduate", "Enrolled"]
pd.set_option("display.max_rows", 200, "display.max_columns", None, "display.max_colwidth", None, "display.precision", 3)

## 1. Pemahaman masalah

Target penelitian adalah `dropout`: 1 untuk status `Dropout`, dan 0 untuk `Graduate` dan `Enrolled`. Prediksi dibuat pada tiga tahap.

| Tahap | Waktu prediksi | Atribut |
| ----- | ---------------- | ------- |
| T1 | Saat masuk | Keuangan, demografi dan keluarga, akademik saat masuk, makroekonomi |
| T2 | Akhir semester 1 | T1 ditambah 6 atribut semester 1 |
| T3 | Akhir semester 2 | T2 ditambah 6 atribut semester 2 |

Rumusan masalah:

1. Seberapa baik Random Forest memprediksi dropout saat masuk, akhir semester 1, dan akhir semester 2, dibandingkan dengan Regresi Logistik dan XGBoost?
2. Seberapa besar performa prediksi berubah pada tiap tahap jika atribut keuangan dihapus?
3. Atribut apa yang paling memengaruhi prediksi pada tiap tahap, dengan dan tanpa atribut keuangan?

Hasil penelitian ini menunjukkan pola prediktif, bukan hubungan sebab akibat.

## 2. Pengumpulan data

Notebook memakai `data/data.csv` jika ada. Jika tidak ada, seperti di Kaggle, notebook mengunduh arsip UCI. Hash `data.csv` selalu diperiksa, jadi seluruh angka di bawah berasal dari file yang sama.

In [3]:
DATA_URL = "https://archive.ics.uci.edu/static/public/697/predict+students+dropout+and+academic+success.zip"
DATA_SHA256 = "3ef126de5cefff26eb11fbb4237f1a1401cb64b488e2f1d598c23cedeb4c45ae"

run_env = "kaggle" if Path("/kaggle/working").exists() else "laptop"
# Jupyter menjalankan kernel dari folder notebooks/, jadi root proyek dicari ke atas.
ROOT = (
    Path("/kaggle/working")
    if run_env == "kaggle"
    else next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "flake.nix").exists()), Path.cwd())
)
DATA_FILE = ROOT / "data" / "data.csv"
ARTIFACTS = ROOT / "artifacts"

if DATA_FILE.exists():
    data_bytes = DATA_FILE.read_bytes()
    data_source = "data/data.csv"
else:
    with urllib.request.urlopen(DATA_URL, timeout=120) as response:
        data_bytes = zipfile.ZipFile(io.BytesIO(response.read())).read("data.csv")
    data_source = "arsip UCI"

# DATA-01
data_sha256 = hashlib.sha256(data_bytes).hexdigest()
if data_sha256 != DATA_SHA256:
    raise ValueError(
        "Hash data.csv berbeda dari yang diharapkan, jadi notebook dihentikan.\n"
        f"  diharapkan: {DATA_SHA256}\n"
        f"  didapat:    {data_sha256}\n"
        "Ambil ulang data.csv dari arsip UCI, lalu jalankan ulang notebook."
    )

# DATA-02: file memakai BOM UTF-8, pemisah `;`, dan akhir baris CRLF. Nama kolom ke-5 diakhiri tab.
raw = pd.read_csv(io.BytesIO(data_bytes), sep=";", encoding="utf-8-sig")
raw.columns = raw.columns.str.strip()
# DATA-06: nomor baris berbasis 0 pada data.csv, bukan identitas orang. Sebagai indeks, row_id tidak ikut menjadi atribut.
raw.index.name = "row_id"

print(f"Lingkungan: {run_env}. Sumber data: {data_source}. Hash cocok: {data_sha256[:12]}...")

Lingkungan: kaggle. Sumber data: arsip UCI. Hash cocok: 3ef126de5cef...


In [4]:
def status_counts(frame):
    return frame["Target"].value_counts().reindex(STATUSES, fill_value=0).to_dict()


# DATA-03: file dikunci dengan hash, jadi cabang pengisian nilai dan penghapusan duplikat tidak diperlukan.
empty_cells = int(raw.isna().sum().sum())
exact_duplicates = int(raw.duplicated().sum())
assert raw.shape == (4424, 37), raw.shape
assert empty_cells == 0, empty_cells
assert exact_duplicates == 0, exact_duplicates
assert status_counts(raw) == {"Dropout": 1421, "Graduate": 2209, "Enrolled": 794}, status_counts(raw)

print(f"File asli: {raw.shape[0]} baris, {raw.shape[1]} kolom, {empty_cells} sel kosong, {exact_duplicates} baris duplikat.")
print("Status:", status_counts(raw))

File asli: 4424 baris, 37 kolom, 0 sel kosong, 0 baris duplikat.
Status: {'Dropout': 1421, 'Graduate': 2209, 'Enrolled': 794}


## 3. Preprocessing

### Mengeluarkan `Course` 171

Seluruh baris dengan 0 mata kuliah di kedua semester berasal dari `Course` 171 (Animation and Multimedia Design), dan sebagian berlabel `Graduate`. Polanya menunjukkan kekosongan data, bukan perilaku mahasiswa, dan mempertahankannya dapat mengacaukan T2 dan T3. Karena itu, seluruh baris program ini dikeluarkan.

In [5]:
# DATA-04
is_171 = raw["Course"] == 171
no_units = (raw["Curricular units 1st sem (enrolled)"] == 0) & (raw["Curricular units 2nd sem (enrolled)"] == 0)

course_171 = {
    "name": "Animation and Multimedia Design",
    "rows": int(is_171.sum()),
    "status_counts": status_counts(raw[is_171]),
    "zero_units_both_semesters": int((is_171 & no_units).sum()),
    "zero_units_graduate": int((is_171 & no_units & (raw["Target"] == "Graduate")).sum()),
}
assert course_171["rows"] == 215, course_171
assert course_171["zero_units_both_semesters"] == 180, course_171
assert course_171["zero_units_graduate"] == 75, course_171
assert not (no_units & ~is_171).any(), "Ada baris tanpa mata kuliah di luar Course 171"

df = raw[~is_171].copy()
assert len(df) == 4209, len(df)
assert df["Course"].nunique() == 16, df["Course"].nunique()

# DATA-05: status asli tetap disimpan untuk stratifikasi dan skenario D.
df["dropout"] = (df["Target"] == "Dropout").astype(int)
assert status_counts(df) == {"Dropout": 1339, "Graduate": 2113, "Enrolled": 757}, status_counts(df)
assert (df["dropout"] == 1).sum() == 1339 and (df["dropout"] == 0).sum() == 2870
assert f"{df['dropout'].mean():.1%}" == "31.8%", df["dropout"].mean()

print(f"Course 171: {course_171}")
print(f"Setelah dikeluarkan: {len(df)} baris, {df['Course'].nunique()} program, dropout {df['dropout'].mean():.1%}.")
print("Status:", status_counts(df))

Course 171: {'name': 'Animation and Multimedia Design', 'rows': 215, 'status_counts': {'Dropout': 82, 'Graduate': 96, 'Enrolled': 37}, 'zero_units_both_semesters': 180, 'zero_units_graduate': 75}
Setelah dikeluarkan: 4209 baris, 16 program, dropout 31.8%.
Status: {'Dropout': 1339, 'Graduate': 2113, 'Enrolled': 757}


### Jenis atribut dan atribut per tahap

Atribut dibagi menjadi 9 nominal, 8 biner, dan 19 numerik. T2 dan T3 menambah 6 atribut semester ke tahap sebelumnya. Skenario tanpa keuangan menghapus `Debtor`, `Tuition fees up to date`, dan `Scholarship holder` dari tiap tahap.

In [6]:
# DATA-07
FINANCIAL = ["Debtor", "Tuition fees up to date", "Scholarship holder"]
NOMINAL = [
    "Marital status",
    "Application mode",
    "Course",
    "Previous qualification",
    "Nacionality",
    "Mother's qualification",
    "Father's qualification",
    "Mother's occupation",
    "Father's occupation",
]
BINARY = [
    "Daytime/evening attendance",
    "Displaced",
    "Educational special needs",
    "Debtor",
    "Tuition fees up to date",
    "Gender",
    "Scholarship holder",
    "International",
]
SEMESTER_FIELDS = ["credited", "enrolled", "evaluations", "approved", "grade", "without evaluations"]
SEM1 = [f"Curricular units 1st sem ({field})" for field in SEMESTER_FIELDS]
SEM2 = [f"Curricular units 2nd sem ({field})" for field in SEMESTER_FIELDS]
NUMERIC = [
    "Application order",
    "Previous qualification (grade)",
    "Admission grade",
    "Age at enrollment",
    "Unemployment rate",
    "Inflation rate",
    "GDP",
    *SEM1,
    *SEM2,
]
assert (len(NOMINAL), len(BINARY), len(NUMERIC)) == (9, 8, 19)
assert sorted(NOMINAL + BINARY + NUMERIC) == sorted(raw.columns.drop("Target")), "Daftar atribut tidak cocok dengan file"
assert df[BINARY].isin([0, 1]).all(axis=None), "Ada atribut biner dengan nilai selain 0 dan 1"

# DATA-08
T1 = [column for column in raw.columns if column not in ["Target", *SEM1, *SEM2]]
WITH_FINANCIAL = {"T1": T1, "T2": T1 + SEM1, "T3": T1 + SEM1 + SEM2}
WITHOUT_FINANCIAL = {stage: [a for a in attrs if a not in FINANCIAL] for stage, attrs in WITH_FINANCIAL.items()}

stage_attributes = {
    stage: {"with": len(WITH_FINANCIAL[stage]), "without": len(WITHOUT_FINANCIAL[stage])} for stage in WITH_FINANCIAL
}
assert stage_attributes == {
    "T1": {"with": 24, "without": 21},
    "T2": {"with": 30, "without": 27},
    "T3": {"with": 36, "without": 33},
}, stage_attributes

pd.DataFrame(stage_attributes).T.rename(columns={"with": "dengan keuangan", "without": "tanpa keuangan"})

,dengan keuangan,tanpa keuangan
T1,24,21
T2,30,27
T3,36,33


### Aturan konsistensi

Nilai `grade` 0 dipertahankan, karena selalu berarti tidak ada mata kuliah yang lulus, bukan nilai nol.

In [7]:
# DATA-09
violations = {}
grade_zero_rows = {}
for semester in ["1st", "2nd"]:
    units = lambda field: df[f"Curricular units {semester} sem ({field})"]
    violations[f"{semester}: approved > enrolled"] = int((units("approved") > units("enrolled")).sum())
    violations[f"{semester}: credited > enrolled"] = int((units("credited") > units("enrolled")).sum())
    violations[f"{semester}: evaluations < approved"] = int((units("evaluations") < units("approved")).sum())
    violations[f"{semester}: (grade = 0) berbeda dari (approved = 0)"] = int(
        ((units("grade") == 0) != (units("approved") == 0)).sum()
    )
    violations[f"{semester}: enrolled < 1"] = int((units("enrolled") < 1).sum())
    grade_zero_rows[semester] = int((units("grade") == 0).sum())

consistency_violations = sum(violations.values())
assert consistency_violations == 0, violations
assert grade_zero_rows == {"1st": 538, "2nd": 690}, grade_zero_rows

print("Baris dengan grade 0:", grade_zero_rows)
pd.Series(violations, name="pelanggaran").to_frame()

Baris dengan grade 0: {'1st': 538, '2nd': 690}


,pelanggaran
1st: approved > enrolled,0
1st: credited > enrolled,0
1st: evaluations < approved,0
1st: (grade = 0) berbeda dari (approved = 0),0
1st: enrolled < 1,0
2nd: approved > enrolled,0
2nd: credited > enrolled,0
2nd: evaluations < approved,0
2nd: (grade = 0) berbeda dari (approved = 0),0
2nd: enrolled < 1,0


### Proporsi dropout per atribut keuangan

Tabel ini memakai seluruh 4.209 baris supaya sama dengan proposal Tabel 5. Angkanya hanya menjadi deskripsi di halaman Dataset dan tidak masuk ke pemodelan.

In [8]:
def category_table(frame, attribute):
    table = pd.crosstab(frame[attribute], frame["Target"]).reindex(columns=STATUSES, fill_value=0)
    table.insert(0, "n", table.sum(axis=1))
    table["dropout_rate"] = (table["Dropout"] / table["n"]).round(6)
    table = table.rename_axis(index="category", columns=None).reset_index()
    table.insert(0, "attribute", attribute)
    return table


financial_rates = pd.concat([category_table(df, attribute) for attribute in FINANCIAL], ignore_index=True)

financial_percent = {
    (row.attribute, row.category): round(row.dropout_rate * 100, 1) for row in financial_rates.itertuples()
}
assert financial_percent == {
    ("Debtor", 0): 27.9,
    ("Debtor", 1): 62.4,
    ("Tuition fees up to date", 0): 87.2,
    ("Tuition fees up to date", 1): 24.4,
    ("Scholarship holder", 0): 38.5,
    ("Scholarship holder", 1): 11.7,
}, financial_percent

financial_rates

,attribute,category,n,Dropout,Graduate,Enrolled,dropout_rate
0,Debtor,0,3735,1043,2017,675,0.279
1,Debtor,1,474,296,96,82,0.624
2,Tuition fees up to date,0,499,435,23,41,0.872
3,Tuition fees up to date,1,3710,904,2090,716,0.244
4,Scholarship holder,0,3162,1217,1312,633,0.385
5,Scholarship holder,1,1047,122,801,124,0.117


### Pembagian data

Pembagian 80:20 distratifikasi menurut tiga status. Pembagian dilakukan sebelum eksplorasi, supaya eksplorasi hanya melihat data latih. Baris uji yang sama dipakai di semua skenario, dan skenario D hanya mengeluarkan `Enrolled` dari kedua himpunan.

In [9]:
# DATA-10
train, test = train_test_split(df, test_size=0.2, stratify=df["Target"], random_state=SEED)
train_d = train[train["Target"] != "Enrolled"]
test_d = test[test["Target"] != "Enrolled"]

split_table = pd.DataFrame(
    {
        "Semua": status_counts(df),
        "Latih": status_counts(train),
        "Uji": status_counts(test),
        "Latih skenario D": status_counts(train_d),
        "Uji skenario D": status_counts(test_d),
    }
).T
split_table.insert(0, "Jumlah", split_table.sum(axis=1))

assert split_table.to_dict("index") == {
    "Semua": {"Jumlah": 4209, "Dropout": 1339, "Graduate": 2113, "Enrolled": 757},
    "Latih": {"Jumlah": 3367, "Dropout": 1071, "Graduate": 1690, "Enrolled": 606},
    "Uji": {"Jumlah": 842, "Dropout": 268, "Graduate": 423, "Enrolled": 151},
    "Latih skenario D": {"Jumlah": 2761, "Dropout": 1071, "Graduate": 1690, "Enrolled": 0},
    "Uji skenario D": {"Jumlah": 691, "Dropout": 268, "Graduate": 423, "Enrolled": 0},
}, split_table
assert train.index.intersection(test.index).empty, "Ada row_id yang masuk data latih dan data uji"

split_table

,Jumlah,Dropout,Graduate,Enrolled
Semua,4209,1339,2113,757
Latih,3367,1071,1690,606
Uji,842,268,423,151
Latih skenario D,2761,1071,1690,0
Uji skenario D,691,268,423,0


### One-hot encoding dan penskalaan

Encoder berada di dalam `Pipeline`, sehingga dihitung ulang di tiap fold dan hanya dari data latih. Atribut nominal memakai one-hot, dan kategori di bawah 1% digabung menjadi satu kolom. Atribut numerik hanya diskalakan untuk Regresi Logistik, dan atribut biner tetap 0 dan 1.

In [10]:
def make_encoder(attributes, scale_numeric):
    # NB-05: model pohon memakai scale_numeric=False, Regresi Logistik memakai scale_numeric=True.
    onehot = OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=0.01, sparse_output=False)
    transformers = [("nominal", onehot, NOMINAL)]
    if scale_numeric:
        transformers.append(("numeric", StandardScaler(), [a for a in attributes if a in NUMERIC]))
    return ColumnTransformer(transformers, remainder="passthrough")

In [11]:
# set_output hanya dipakai di sel ini, supaya kolom hasil encoder dapat diperiksa menurut nama.
X_t3 = train[WITH_FINANCIAL["T3"]]
tree_encoder = make_encoder(WITH_FINANCIAL["T3"], scale_numeric=False).set_output(transform="pandas").fit(X_t3)
linear_encoder = make_encoder(WITH_FINANCIAL["T3"], scale_numeric=True).set_output(transform="pandas").fit(X_t3)
tree_out, linear_out = tree_encoder.transform(X_t3), linear_encoder.transform(X_t3)

onehot_rows = []
for attribute in NOMINAL:
    columns = [column for column in tree_out.columns if column.startswith(f"nominal__{attribute}_")]
    counts = train[attribute].value_counts()
    frequent = int((counts >= 0.01 * len(train)).sum())
    assert len(columns) == frequent + (frequent < len(counts)), f"{attribute}: jumlah kolom one-hot tidak sesuai min_frequency"
    assert (tree_out[columns].sum(axis=1) == 1).all(), f"{attribute}: tiap baris harus tepat satu kolom one-hot"
    onehot_rows.append({"attribute": attribute, "kategori_di_data_latih": len(counts), "kolom_one_hot": len(columns)})

tree_numeric = tree_out[[f"remainder__{attribute}" for attribute in NUMERIC]]
assert (tree_numeric.to_numpy() == train[NUMERIC].to_numpy()).all(), "Model pohon tidak boleh menskalakan atribut numerik"
linear_binary = linear_out[[f"remainder__{attribute}" for attribute in BINARY]]
assert (linear_binary.to_numpy() == train[BINARY].to_numpy()).all(), "Atribut biner tidak boleh diskalakan"
linear_numeric = linear_out[[f"numeric__{attribute}" for attribute in NUMERIC]]
assert (linear_numeric.mean().abs() < 1e-9).all(), "Atribut numerik Regresi Logistik harus berpusat di 0"
assert ((linear_numeric.std(ddof=0) - 1).abs() < 1e-9).all(), "Atribut numerik Regresi Logistik harus bersimpangan baku 1"

unseen = tree_encoder.transform(X_t3.head(1).assign(Course=-1))
assert unseen["nominal__Course_infrequent_sklearn"].iloc[0] == 1, "Kategori baru harus masuk ke kolom kategori jarang"

pd.DataFrame(onehot_rows)

,attribute,kategori_di_data_latih,kolom_one_hot
0,Marital status,6,4
1,Application mode,18,10
2,Course,16,16
3,Previous qualification,17,6
4,Nacionality,17,2
5,Mother's qualification,28,9
6,Father's qualification,31,8
7,Mother's occupation,31,11
8,Father's occupation,40,13


## 4. Eksplorasi data

Seluruh tabel pada bagian ini hanya memakai `train`. Data uji tidak dipakai, supaya tidak memengaruhi rancangan model.

### Distribusi atribut numerik per status

In [12]:
numeric_by_status = (
    train.melt(id_vars="Target", value_vars=NUMERIC, var_name="attribute")
    .groupby(["attribute", "Target"])["value"]
    .describe()[["mean", "25%", "50%", "75%"]]
    .rename(columns={"25%": "q1", "50%": "median", "75%": "q3"})
    .round(6)
    .rename_axis(index=["attribute", "status"])
    .reset_index()
)

numeric_by_status.pivot(index="attribute", columns="status", values=["q1", "median", "q3"]).reindex(
    index=NUMERIC, columns=pd.MultiIndex.from_product([["q1", "median", "q3"], STATUSES])
)

q1                    \
                                               Dropout Graduate Enrolled   
attribute                                                                  
Application order                                  1.0    1.000      1.0   
Previous qualification (grade)                   122.5  125.000    122.0   
Admission grade                                  115.0  119.400    116.5   
Age at enrollment                                 19.0   18.000     19.0   
Unemployment rate                                  9.4    9.400      9.4   
Inflation rate                                     0.3    0.300      0.3   
GDP                                               -1.7   -1.700     -1.7   
Curricular units 1st sem (credited)                0.0    0.000      0.0   
Curricular units 1st sem (enrolled)                5.0    6.000      5.0   
Curricular units 1st sem (evaluations)             6.0    7.000      8.0   
Curricular units 1st sem (approved)                0.0    5.000      3.0   
Curricular units 1st sem (grade)                   0.0   12.200     11.0   
Curricular units 1st sem (without evaluations)     0.0    0.000      0.0   
Curricular units 2nd sem (credited)                0.0    0.000      0.0   
Curricular units 2nd sem (enrolled)                5.0    6.000      5.0   
Curricular units 2nd sem (evaluations)             5.0    6.000      8.0   
Curricular units 2nd sem (approved)                0.0    5.000      3.0   
Curricular units 2nd sem (grade)                   0.0   12.232     11.0   
Curricular units 2nd sem (without evaluations)     0.0    0.000      0.0   

                                                median                    \
                                               Dropout Graduate Enrolled   
attribute                                                                  
Application order                                 1.00    1.000     1.00   
Previous qualification (grade)                  133.10  133.100   130.00   
Admission grade                                 123.60  127.200   123.50   
Age at enrollment                                24.00   19.000    20.00   
Unemployment rate                                11.10   11.100    11.10   
Inflation rate                                    1.40    0.600     1.40   
GDP                                               0.32    0.790     0.32   
Curricular units 1st sem (credited)               0.00    0.000     0.00   
Curricular units 1st sem (enrolled)               6.00    6.000     6.00   
Curricular units 1st sem (evaluations)            8.00    8.000     9.00   
Curricular units 1st sem (approved)               2.00    6.000     5.00   
Curricular units 1st sem (grade)                 11.00   13.059    12.00   
Curricular units 1st sem (without evaluations)    0.00    0.000     0.00   
Curricular units 2nd sem (credited)               0.00    0.000     0.00   
Curricular units 2nd sem (enrolled)               6.00    6.000     6.00   
Curricular units 2nd sem (evaluations)            8.00    8.000     9.00   
Curricular units 2nd sem (approved)               1.00    6.000     4.00   
Curricular units 2nd sem (grade)                 10.00   13.035    12.00   
Curricular units 2nd sem (without evaluations)    0.00    0.000     0.00   

                                                    q3                    
                                               Dropout Graduate Enrolled  
attribute                                                                 
Application order                                 2.00    2.000    2.000  
Previous qualification (grade)                  137.50  141.000  140.000  
Admission grade                                 132.95  135.900  132.400  
Age at enrollment                                31.00   21.000   24.000  
Unemployment rate                                13.90   13.900   12.700  
Inflation rate                                    2.60    2.600    2.600  
GDP                            

### Proporsi dropout per kategori

Tabel lengkap untuk semua kategori disimpan di `dataset_summary.json`. Kode kategori mengikuti tabel variabel UCI. Kategori kecil menghasilkan proporsi yang tidak stabil, jadi ringkasan atribut nominal hanya memakai kategori dengan minimal 1% baris latih.

In [13]:
category_rates = pd.concat([category_table(train, attribute) for attribute in NOMINAL + BINARY], ignore_index=True)

binary_rates = category_rates[category_rates["attribute"].isin(BINARY)]
binary_rates.pivot(index="attribute", columns="category", values=["n", "dropout_rate"]).reindex(BINARY).astype(
    {("n", 0): int, ("n", 1): int}
)

n       dropout_rate       
category                       0     1            0      1
attribute                                                 
Daytime/evening attendance   394  2973        0.439  0.302
Displaced                   1577  1790        0.370  0.272
Educational special needs   3330    37        0.318  0.324
Debtor                      2993   374        0.279  0.628
Tuition fees up to date      390  2977        0.872  0.246
Gender                      2219  1148        0.251  0.447
Scholarship holder          2518   849        0.386  0.118
International               3290    77        0.319  0.286

In [14]:
nominal_rates = category_rates[category_rates["attribute"].isin(NOMINAL)]
frequent_rates = nominal_rates[nominal_rates["n"] >= 0.01 * len(train)]

nominal_overview = frequent_rates.groupby("attribute").agg(
    kategori_minimal_1_persen=("category", "size"),
    dropout_terendah=("dropout_rate", "min"),
    dropout_tertinggi=("dropout_rate", "max"),
)
nominal_overview.insert(0, "kategori_di_data_latih", nominal_rates.groupby("attribute").size())
nominal_overview.reindex(NOMINAL)

,kategori_di_data_latih,kategori_minimal_1_persen,dropout_terendah,dropout_tertinggi
attribute,,,,
Marital status,6,3,0.297,0.474
Application mode,18,9,0.189,0.630
Course,16,15,0.149,0.596
Previous qualification,17,5,0.281,0.662
Nacionality,17,1,0.319,0.319
Mother's qualification,28,8,0.146,0.755
Father's qualification,31,7,0.225,0.734
Mother's occupation,31,10,0.260,0.768
Father's occupation,40,12,0.255,0.765


In [15]:
course_rates = frequent_rates[frequent_rates["attribute"] == "Course"].drop(columns="attribute")
course_rates.sort_values("dropout_rate", ascending=False, kind="stable").reset_index(drop=True)

,category,n,Dropout,Graduate,Enrolled,dropout_rate
0,9130,114,68,32,14,0.596
1,9119,135,70,10,55,0.519
2,9991,222,113,61,48,0.509
3,9853,158,67,51,40,0.424
4,9254,186,74,80,32,0.398
5,9003,163,64,70,29,0.393
6,9556,72,27,32,13,0.375
7,9670,226,82,105,39,0.363
8,8014,172,60,97,15,0.349
9,9147,296,102,105,89,0.345


### Korelasi antaratribut

Korelasi Pearson hanya dihitung pada atribut numerik dan biner, karena kode atribut nominal tidak memiliki urutan.

In [16]:
corr = train[NUMERIC + BINARY].corr()
correlations = pd.DataFrame(
    [(a, b, corr.loc[a, b]) for a, b in itertools.combinations(corr.columns, 2)],
    columns=["attribute_a", "attribute_b", "r"],
)
assert correlations["r"].notna().all(), "Ada atribut tanpa variasi pada data latih"

top_correlations = (
    correlations.sort_values("r", key=abs, ascending=False, kind="stable").head(20).round(6).reset_index(drop=True)
)
top_correlations

,attribute_a,attribute_b,r
0,Curricular units 1st sem (credited),Curricular units 2nd sem (credited),0.942
1,Curricular units 1st sem (enrolled),Curricular units 2nd sem (enrolled),0.914
2,Curricular units 1st sem (approved),Curricular units 2nd sem (approved),0.891
3,Curricular units 1st sem (credited),Curricular units 1st sem (enrolled),0.860
4,Curricular units 1st sem (enrolled),Curricular units 2nd sem (credited),0.841
5,Curricular units 1st sem (grade),Curricular units 2nd sem (grade),0.807
6,Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),0.773
7,Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),0.746
8,Curricular units 1st sem (evaluations),Curricular units 2nd sem (evaluations),0.738
9,Curricular units 1st sem (enrolled),Curricular units 1st sem (approved),0.734


In [17]:
first_stage = {attribute: "T1" for attribute in T1} | {a: "T2" for a in SEM1} | {a: "T3" for a in SEM2}

target_correlations = (
    train[NUMERIC + BINARY]
    .corrwith(train["dropout"])
    .rename("r")
    .rename_axis("attribute")
    .reset_index()
    .assign(stage=lambda table: table["attribute"].map(first_stage))
    .sort_values("r", key=abs, ascending=False, kind="stable")
    .round(6)
    .reset_index(drop=True)
)
target_correlations

,attribute,r,stage
0,Curricular units 2nd sem (grade),-0.626,T3
1,Curricular units 2nd sem (approved),-0.612,T3
2,Curricular units 1st sem (grade),-0.533,T2
3,Curricular units 1st sem (approved),-0.513,T2
4,Tuition fees up to date,-0.430,T1
5,Age at enrollment,0.269,T1
6,Scholarship holder,-0.250,T1
7,Debtor,0.235,T1
8,Gender,0.199,T1
9,Curricular units 2nd sem (evaluations),-0.162,T3


### Ringkasan eksplorasi

Angka berikut menjadi dasar catatan penafsiran di bawahnya.

In [18]:
def strongest(stage, k=3):
    rows = target_correlations[target_correlations["stage"] == stage].head(k)
    return ", ".join(f"{row.attribute} ({row.r:+.2f})" for row in rows.itertuples())


high_pairs = correlations[correlations["r"].abs() >= 0.7]
pairs_across_semesters = high_pairs[
    high_pairs["attribute_a"].isin(SEM1) & high_pairs["attribute_b"].isin(SEM2)
]
rate_by_course = course_rates.set_index("category")["dropout_rate"]

print(f"Data latih: {len(train)} baris, dropout {train['dropout'].mean():.1%}.")
for stage in ["T1", "T2", "T3"]:
    print(f"Korelasi terkuat dengan dropout, atribut baru pada {stage}: {strongest(stage)}.")
for semester in ["1st", "2nd"]:
    grade = f"Curricular units {semester} sem (grade)"
    passed = train[train[grade] > 0]
    print(
        f"{grade}: r dengan dropout {train[grade].corr(train['dropout']):+.2f}, "
        f"tanpa baris grade 0 {passed[grade].corr(passed['dropout']):+.2f}. "
        f"Dropout {train.loc[train[grade] == 0, 'dropout'].mean():.1%} pada grade 0, "
        f"{passed['dropout'].mean():.1%} pada grade di atas 0."
    )
print(
    f"Pasangan atribut dengan |r| >= 0.7: {len(high_pairs)}, "
    f"{len(pairs_across_semesters)} di antaranya antara atribut semester 1 dan semester 2."
)
print(
    f"Proporsi dropout per Course: tertinggi {rate_by_course.max():.1%} (kode {rate_by_course.idxmax()}), "
    f"terendah {rate_by_course.min():.1%} (kode {rate_by_course.idxmin()})."
)

Data latih: 3367 baris, dropout 31.8%.
Korelasi terkuat dengan dropout, atribut baru pada T1: Tuition fees up to date (-0.43), Age at enrollment (+0.27), Scholarship holder (-0.25).
Korelasi terkuat dengan dropout, atribut baru pada T2: Curricular units 1st sem (grade) (-0.53), Curricular units 1st sem (approved) (-0.51), Curricular units 1st sem (enrolled) (-0.13).
Korelasi terkuat dengan dropout, atribut baru pada T3: Curricular units 2nd sem (grade) (-0.63), Curricular units 2nd sem (approved) (-0.61), Curricular units 2nd sem (evaluations) (-0.16).
Curricular units 1st sem (grade): r dengan dropout -0.53, tanpa baris grade 0 -0.23. Dropout 91.1% pada grade 0, 22.7% pada grade di atas 0.
Curricular units 2nd sem (grade): r dengan dropout -0.63, tanpa baris grade 0 -0.22. Dropout 94.0% pada grade 0, 19.2% pada grade di atas 0.
Pasangan atribut dengan |r| >= 0.7: 11, 7 di antaranya antara atribut semester 1 dan semester 2.
Proporsi dropout per Course: tertinggi 59.6% (kode 9130), tere

**Catatan penafsiran**

- Angka di atas adalah korelasi linear satu atribut pada data latih. Angka ini bukan perkiraan performa model dan bukan hubungan sebab akibat.
- Pada T1, korelasi terkuat dimiliki `Tuition fees up to date`, yaitu atribut keuangan. Waktu pencatatannya tidak dirinci pada sumber data, sehingga rumusan masalah 2 perlu mengukur seberapa jauh T1 bergantung pada atribut ini.
- Sebagian besar korelasi `grade` berasal dari baris `grade` 0, yaitu mahasiswa tanpa mata kuliah yang lulus, dan hampir semua baris ini berakhir dropout. File publik tidak memiliki tanggal keluar, jadi nilai 0 bisa saja mencatat mahasiswa yang sudah berhenti kuliah. Hal ini sejalan dengan keterbatasan rancangan bahwa T3 mungkin terlalu optimistis.
- Atribut semester 1 dan semester 2 saling berkorelasi kuat. Pada T3, kontribusi SHAP dapat terbagi di antara atribut yang serupa, jadi peringkat atribut semester 1 pada T3 perlu dibaca hati-hati. Multikolinearitas ini juga menjadi alasan Regresi Logistik memakai regularisasi L2.
- Banyak kategori nominal berada di bawah 1% dan akan digabung saat one-hot encoding. `Nacionality` hanya menyisakan satu kategori di atas batas itu.

Eksplorasi ini tidak mengubah rancangan. Seluruh 36 atribut, ruang pencarian, dan ambang keputusan tetap mengikuti proposal.

### Simpan ringkasan

Angka pemeriksaan data dan eksplorasi disimpan ke `dataset_summary.json` untuk halaman Dataset pada aplikasi. Fungsi `write_artifact` di sel ini juga menulis artefak JSON lainnya.

In [19]:
dataset_summary = {
    "status_counts": {"file": status_counts(raw), "after_exclusion": status_counts(df)},
    "course_171": course_171,
    "quality": {
        "rows": raw.shape[0],
        "columns": raw.shape[1],
        "empty_cells": empty_cells,
        "exact_duplicates": exact_duplicates,
        "rows_after_exclusion": len(df),
        "programs_after_exclusion": int(df["Course"].nunique()),
        "grade_zero_rows": grade_zero_rows,
        "consistency_violations": consistency_violations,
    },
    "stage_attributes": stage_attributes,
    "financial_dropout_rates": {"source": "all", "n_rows": len(df), "rows": financial_rates.to_dict("records")},
    "exploration": {
        "source": "train",
        "n_rows": len(train),
        "dropout_rate": round(float(train["dropout"].mean()), 6),
        "numeric_by_status": numeric_by_status.to_dict("records"),
        "category_rates": category_rates.to_dict("records"),
        "top_correlations": top_correlations.to_dict("records"),
        "target_correlations": target_correlations.to_dict("records"),
    },
}


def write_artifact(name, data, indent=2):
    ARTIFACTS.mkdir(parents=True, exist_ok=True)
    text = json.dumps(data, indent=indent, ensure_ascii=False, allow_nan=False) + "\n"
    (ARTIFACTS / name).write_text(text, encoding="utf-8")
    print(f"artifacts/{name} ditulis, {len(text) / 1024:.0f} KB.")


write_artifact("dataset_summary.json", dataset_summary)

artifacts/dataset_summary.json ditulis, 62 KB.


## 5. Pemodelan

### Baseline Regresi Logistik

Baseline memakai atribut skenario A pada tiap tahap, dengan konfigurasi dan pencarian parameter yang sama dengan model akhir. Seluruh skor di bawah adalah PR-AUC dari 5-fold CV pada data latih. Data uji belum dinilai pada tahap ini, jadi skor ini bukan hasil penelitian.

In [20]:
CV = StratifiedKFold(5, shuffle=True, random_state=SEED)


def tune(classifier, attributes, scale_numeric, space, n_iter, frame=train):
    # NB-07: pencarian parameter hanya melihat data latih.
    pipeline = Pipeline([("enc", make_encoder(attributes, scale_numeric)), ("clf", classifier)])
    search = RandomizedSearchCV(
        pipeline, space, n_iter=n_iter, scoring="average_precision", n_jobs=-1, cv=CV, random_state=SEED
    )
    started = time.perf_counter()
    search.fit(frame[attributes], frame["dropout"])
    timings["tuning"] += time.perf_counter() - started
    return search


LR_SPACE = {"clf__C": [0.01, 0.1, 1, 10]}


def tune_lr(attributes):
    # NB-06: argumen penalty tidak diisi, karena L2 sudah menjadi bawaan.
    classifier = LogisticRegression(class_weight="balanced", max_iter=5000)
    return tune(classifier, attributes, scale_numeric=True, space=LR_SPACE, n_iter=4)


def check_search(model_id, attributes, n_iter):
    search = searches[model_id]
    model = search.best_estimator_
    tried = {json.dumps(params, sort_keys=True) for params in search.cv_results_["params"]}
    is_lr = isinstance(model["clf"], LogisticRegression)
    assert list(model.feature_names_in_) == attributes, f"{model_id}: atribut masukan tidak sesuai"
    assert len(tried) == n_iter, f"{model_id}: harus mencoba {n_iter} kombinasi berbeda"
    assert search.n_splits_ == 5, f"{model_id}: harus memakai 5 fold"
    assert ("numeric" in model["enc"].named_transformers_) == is_lr, f"{model_id}: hanya Regresi Logistik yang menskalakan"
    assert not is_lr or model["clf"].n_iter_.max() < model["clf"].max_iter, f"{model_id}: Regresi Logistik belum konvergen"


searches = {f"A_{stage}_lr": tune_lr(attributes) for stage, attributes in WITH_FINANCIAL.items()}
for stage, attributes in WITH_FINANCIAL.items():
    check_search(f"A_{stage}_lr", attributes, n_iter=4)

print(f"Patokan terendah PR-AUC pada data latih: {train['dropout'].mean():.3f}.")
pd.DataFrame(
    [
        {
            "model_id": model_id,
            "atribut": search.n_features_in_,
            "kolom_setelah_encoding": len(search.best_estimator_["enc"].get_feature_names_out()),
            "C_terbaik": search.best_params_["clf__C"],
            "cv_pr_auc": round(float(search.best_score_), 6),
            "std_antarfold": round(float(search.cv_results_["std_test_score"][search.best_index_]), 6),
        }
        for model_id, search in searches.items()
    ]
)

Patokan terendah PR-AUC pada data latih: 0.318.


,model_id,atribut,kolom_setelah_encoding,C_terbaik,cv_pr_auc,std_antarfold
0,A_T1_lr,24,94,1,0.739,0.018
1,A_T2_lr,30,100,1,0.869,0.005
2,A_T3_lr,36,106,1,0.894,0.006


In [21]:
# Skor rata-rata tiap nilai C, untuk melihat kepekaan baseline terhadap parameter.
pd.DataFrame(
    {
        model_id: pd.Series(search.cv_results_["mean_test_score"], index=search.cv_results_["param_clf__C"].tolist())
        for model_id, search in searches.items()
    }
).rename_axis("C").sort_index()

,A_T1_lr,A_T2_lr,A_T3_lr
C,,,
0.01,0.700,0.845,0.875
0.10,0.737,0.866,0.893
1.00,0.739,0.869,0.894
10.00,0.738,0.867,0.892


**Catatan penafsiran.** Pada ketiga tahap, skor baseline jauh di atas patokan terendah dan naik dari T1 ke T3, dengan kenaikan terbesar dari T1 ke T2. Perbandingan antartahap dan antaralgoritma baru dinilai pada data uji, setelah seluruh parameter skenario ditetapkan. Skor T3 tetap perlu dibaca dengan keterbatasan bahwa tahap ini mungkin terlalu optimistis.

### Skenario A

Skenario A memakai seluruh atribut tiap tahap. Random Forest dan XGBoost dicari dengan `n_iter=20` pada data latih, sedangkan Regresi Logistik sudah dicari pada baseline. Parameter ketiga algoritma ditetapkan sepenuhnya dari CV, sebelum data uji dinilai.

In [22]:
# NB-06 dan NB-07
RF_SPACE = {
    "clf__n_estimators": [300, 500],
    "clf__max_depth": [None, 10, 20],
    "clf__min_samples_leaf": [1, 2, 5],
    "clf__max_features": ["sqrt", "log2"],
}
XGB_SPACE = {
    "clf__n_estimators": [200, 400],
    "clf__max_depth": [3, 5, 7],
    "clf__learning_rate": [0.03, 0.1],
    "clf__subsample": [0.8, 1.0],
    "clf__colsample_bytree": [0.8, 1.0],
}


def tune_rf(attributes, frame=train):
    classifier = RandomForestClassifier(class_weight="balanced", random_state=SEED, n_jobs=1)
    return tune(classifier, attributes, scale_numeric=False, space=RF_SPACE, n_iter=20, frame=frame)


def tune_xgb(attributes):
    positive = int(train["dropout"].sum())
    classifier = XGBClassifier(
        scale_pos_weight=(len(train) - positive) / positive,
        tree_method="hist",
        eval_metric="logloss",
        random_state=SEED,
        n_jobs=1,
    )
    return tune(classifier, attributes, scale_numeric=False, space=XGB_SPACE, n_iter=20)


for stage, attributes in WITH_FINANCIAL.items():
    searches[f"A_{stage}_rf"] = tune_rf(attributes)
    searches[f"A_{stage}_xgb"] = tune_xgb(attributes)

In [23]:
SCENARIO_A = [f"A_{stage}_{algorithm}" for stage in WITH_FINANCIAL for algorithm in ["rf", "lr", "xgb"]]
assert sorted(searches) == sorted(SCENARIO_A), sorted(searches)
assert searches["A_T1_xgb"].best_estimator_["clf"].scale_pos_weight == 2296 / 1071, "scale_pos_weight harus negatif/positif"

for stage, attributes in WITH_FINANCIAL.items():
    for algorithm in ["rf", "xgb"]:
        check_search(f"A_{stage}_{algorithm}", attributes, n_iter=20)


def best_params(search):
    return {name.removeprefix("clf__"): value for name, value in search.best_params_.items()}


def cv_table(model_ids):
    rows = []
    for model_id in model_ids:
        search = searches[model_id]
        rows.append(
            {
                "model_id": model_id,
                "cv_pr_auc": round(float(search.best_score_), 6),
                "std_antarfold": round(float(search.cv_results_["std_test_score"][search.best_index_]), 6),
                "parameter_terbaik": best_params(search),
            }
        )
    return pd.DataFrame(rows)


print(f"Patokan terendah PR-AUC pada data latih: {train['dropout'].mean():.3f}.")
cv_table(SCENARIO_A)

Patokan terendah PR-AUC pada data latih: 0.318.


,model_id,cv_pr_auc,std_antarfold,parameter_terbaik
0,A_T1_rf,0.737,0.017,"{'n_estimators': 500, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': None}"
1,A_T1_lr,0.739,0.018,{'C': 1}
2,A_T1_xgb,0.752,0.008,"{'subsample': 1.0, 'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.1, 'colsample_bytree': 0.8}"
3,A_T2_rf,0.862,0.014,"{'n_estimators': 300, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': 20}"
4,A_T2_lr,0.869,0.005,{'C': 1}
5,A_T2_xgb,0.869,0.011,"{'subsample': 1.0, 'n_estimators': 400, 'max_depth': 3, 'learning_rate': 0.03, 'colsample_bytree': 0.8}"
6,A_T3_rf,0.885,0.017,"{'n_estimators': 500, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': None}"
7,A_T3_lr,0.894,0.006,{'C': 1}
8,A_T3_xgb,0.893,0.010,"{'subsample': 0.8, 'n_estimators': 400, 'max_depth': 5, 'learning_rate': 0.03, 'colsample_bytree': 0.8}"


### Skenario B, C, dan D

Skenario B menghapus ketiga atribut keuangan pada tiap tahap, dengan ketiga algoritma. Skenario C menghapus satu atribut keuangan secara bergantian pada T1, dengan Random Forest. Skenario D memakai seluruh atribut tiap tahap dengan Random Forest, dan mengeluarkan `Enrolled` dari data latih dan data uji, sebagai pembanding dengan penelitian yang mengeluarkan kelas itu. Konfigurasi dan ruang pencarian sama dengan skenario A, jadi perbedaan hasil hanya berasal dari atribut dan baris yang dipakai.

In [24]:
SCENARIO_B = [f"B_{stage}_{algorithm}" for stage in WITHOUT_FINANCIAL for algorithm in ["rf", "lr", "xgb"]]
for stage, attributes in WITHOUT_FINANCIAL.items():
    searches[f"B_{stage}_rf"] = tune_rf(attributes)
    searches[f"B_{stage}_lr"] = tune_lr(attributes)
    searches[f"B_{stage}_xgb"] = tune_xgb(attributes)

C_REMOVED = {
    "C_T1_rf_no-debtor": "Debtor",
    "C_T1_rf_no-tuition": "Tuition fees up to date",
    "C_T1_rf_no-scholarship": "Scholarship holder",
}
C_ATTRIBUTES = {model_id: [a for a in T1 if a != removed] for model_id, removed in C_REMOVED.items()}
for model_id, attributes in C_ATTRIBUTES.items():
    searches[model_id] = tune_rf(attributes)

SCENARIO_D = [f"D_{stage}_rf" for stage in WITH_FINANCIAL]
for stage, attributes in WITH_FINANCIAL.items():
    searches[f"D_{stage}_rf"] = tune_rf(attributes, frame=train_d)

In [25]:
ALL_MODELS = SCENARIO_A + SCENARIO_B + list(C_REMOVED) + SCENARIO_D
assert len(ALL_MODELS) == 24 and sorted(searches) == sorted(ALL_MODELS), sorted(searches)
assert set(C_REMOVED.values()) == set(FINANCIAL), C_REMOVED

for stage, attributes in WITHOUT_FINANCIAL.items():
    check_search(f"B_{stage}_rf", attributes, n_iter=20)
    check_search(f"B_{stage}_lr", attributes, n_iter=4)
    check_search(f"B_{stage}_xgb", attributes, n_iter=20)
for model_id, attributes in C_ATTRIBUTES.items():
    assert len(attributes) == 23, f"{model_id}: harus memakai 23 atribut"
    check_search(model_id, attributes, n_iter=20)
for stage, attributes in WITH_FINANCIAL.items():
    check_search(f"D_{stage}_rf", attributes, n_iter=20)


def refit_rows(model_id):
    # Panjang sampel bootstrap Random Forest menunjukkan jumlah baris yang benar-benar dipakai refit.
    return len(searches[model_id].best_estimator_["clf"].estimators_samples_[0])


assert all(refit_rows(model_id) == 2761 for model_id in SCENARIO_D), "Skenario D harus dilatih tanpa Enrolled"
assert refit_rows("B_T1_rf") == refit_rows("C_T1_rf_no-debtor") == 3367, "Skenario B dan C harus memakai seluruh data latih"

print(
    f"Patokan terendah PR-AUC pada data latih: {train['dropout'].mean():.3f}, "
    f"atau {train_d['dropout'].mean():.3f} untuk skenario D."
)
cv_table(SCENARIO_B + list(C_REMOVED) + SCENARIO_D)

Patokan terendah PR-AUC pada data latih: 0.318, atau 0.388 untuk skenario D.


,model_id,cv_pr_auc,std_antarfold,parameter_terbaik
0,B_T1_rf,0.613,0.014,"{'n_estimators': 500, 'min_samples_leaf': 2, 'max_features': 'log2', 'max_depth': None}"
1,B_T1_lr,0.619,0.023,{'C': 1}
2,B_T1_xgb,0.630,0.022,"{'subsample': 1.0, 'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.1, 'colsample_bytree': 0.8}"
3,B_T2_rf,0.826,0.014,"{'n_estimators': 500, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': None}"
4,B_T2_lr,0.836,0.011,{'C': 1}
5,B_T2_xgb,0.837,0.011,"{'subsample': 0.8, 'n_estimators': 400, 'max_depth': 3, 'learning_rate': 0.03, 'colsample_bytree': 1.0}"
6,B_T3_rf,0.863,0.018,"{'n_estimators': 500, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': 20}"
7,B_T3_lr,0.874,0.010,{'C': 0.1}
8,B_T3_xgb,0.869,0.015,"{'subsample': 1.0, 'n_estimators': 400, 'max_depth': 3, 'learning_rate': 0.03, 'colsample_bytree': 0.8}"
9,C_T1_rf_no-debtor,0.735,0.016,"{'n_estimators': 300, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'max_depth': 20}"


## 6. Evaluasi

Metrik dihitung pada data uji setelah parameter ditetapkan. Mahasiswa ditandai jika `risk >= 0.5`, dan metrik ambang dihitung dari aturan itu, karena `.predict()` memperlakukan nilai tepat 0,5 secara berbeda. Sel berikut menguji `evaluate` pada angka hitungan tangan, tanpa menyentuh data uji.

In [26]:
def evaluate(model, frame):
    # NB-09
    y_true = frame["dropout"]
    risk = model.predict_proba(frame[list(model.feature_names_in_)])[:, 1]
    flagged = (risk >= THRESHOLD).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, flagged, labels=[0, 1]).ravel().tolist()
    return {
        "pr_auc": round(float(average_precision_score(y_true, risk)), 6),
        "recall": round(float(recall_score(y_true, flagged, zero_division=0)), 6),
        "precision": round(float(precision_score(y_true, flagged, zero_division=0)), 6),
        "f1": round(float(f1_score(y_true, flagged, zero_division=0)), 6),
        "roc_auc": round(float(roc_auc_score(y_true, risk)), 6),
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
    }


class FixedRisk:
    feature_names_in_ = ["x"]

    def __init__(self, risk):
        self.risk = np.array(risk)

    def predict_proba(self, X):
        return np.column_stack([1 - self.risk, self.risk])


# Baris dengan risk tepat 0,5 berlabel 0, jadi harus terhitung sebagai fp.
evaluate_check = evaluate(FixedRisk([0.9, 0.5, 0.4, 0.1]), pd.DataFrame({"x": [0, 0, 0, 0], "dropout": [1, 0, 1, 0]}))
assert evaluate_check == {
    "pr_auc": 0.833333,
    "recall": 0.5,
    "precision": 0.5,
    "f1": 0.5,
    "roc_auc": 0.75,
    "tp": 1,
    "fp": 1,
    "fn": 1,
    "tn": 1,
}, evaluate_check

### Seluruh skenario pada data uji

Sel ini adalah penilaian data uji. Parameter dan ambang di atas tidak boleh diubah berdasarkan hasilnya.

In [27]:
FINANCIAL_LABEL = {"A": "with", "B": "without", "C": "drop_one", "D": "with"}
# NB-08: jumlah atribut per tahap, baris latih, dan baris uji tiap skenario.
EXPECTED_SHAPE = {
    "A": ({"T1": 24, "T2": 30, "T3": 36}, 3367, 842),
    "B": ({"T1": 21, "T2": 27, "T3": 33}, 3367, 842),
    "C": ({"T1": 23}, 3367, 842),
    "D": ({"T1": 24, "T2": 30, "T3": 36}, 2761, 691),
}

metrics = []
for model_id in ALL_MODELS:
    scenario, stage, algorithm = model_id.split("_")[:3]
    search = searches[model_id]
    train_frame, test_frame = (train_d, test_d) if scenario == "D" else (train, test)
    metrics.append(
        {
            "model_id": model_id,
            "scenario": scenario,
            "stage": stage,
            "algorithm": algorithm,
            "financial": FINANCIAL_LABEL[scenario],
            "removed": C_REMOVED.get(model_id),
            "n_train": len(train_frame),
            "n_test": len(test_frame),
            **evaluate(search.best_estimator_, test_frame),
            "cv_pr_auc": round(float(search.best_score_), 6),
            "best_params": best_params(search),
        }
    )

assert len(metrics) == 24, len(metrics)
for entry in metrics:
    attribute_counts, n_train, n_test = EXPECTED_SHAPE[entry["scenario"]]
    assert searches[entry["model_id"]].n_features_in_ == attribute_counts[entry["stage"]], entry["model_id"]
    assert (entry["n_train"], entry["n_test"]) == (n_train, n_test), entry["model_id"]
    assert entry["tp"] + entry["fn"] == 268 and entry["fp"] + entry["tn"] == n_test - 268, entry["model_id"]

write_artifact("metrics.json", metrics)
print(f"Patokan terendah PR-AUC pada data uji: {test['dropout'].mean():.3f}, atau {test_d['dropout'].mean():.3f} untuk skenario D.")
pd.DataFrame(metrics).drop(columns=["scenario", "stage", "algorithm", "financial", "removed", "n_train", "n_test", "best_params"])

artifacts/metrics.json ditulis, 12 KB.
Patokan terendah PR-AUC pada data uji: 0.318, atau 0.388 untuk skenario D.


,model_id,pr_auc,recall,precision,f1,roc_auc,tp,fp,fn,tn,cv_pr_auc
0,A_T1_rf,0.734,0.638,0.679,0.658,0.826,171,81,97,493,0.737
1,A_T1_lr,0.744,0.713,0.606,0.655,0.822,191,124,77,450,0.739
2,A_T1_xgb,0.759,0.701,0.648,0.674,0.835,188,102,80,472,0.752
3,A_T2_rf,0.847,0.705,0.815,0.756,0.896,189,43,79,531,0.862
4,A_T2_lr,0.852,0.799,0.746,0.771,0.900,214,73,54,501,0.869
5,A_T2_xgb,0.867,0.772,0.775,0.774,0.907,207,60,61,514,0.869
6,A_T3_rf,0.888,0.772,0.838,0.804,0.927,207,40,61,534,0.885
7,A_T3_lr,0.879,0.825,0.775,0.799,0.919,221,64,47,510,0.894
8,A_T3_xgb,0.889,0.795,0.826,0.810,0.926,213,45,55,529,0.893
9,B_T1_rf,0.589,0.601,0.567,0.583,0.753,161,123,107,451,0.613


In [28]:
for stage in WITH_FINANCIAL:
    scenario_a = (entry for entry in metrics if entry["scenario"] == "A" and entry["stage"] == stage)
    ranked = sorted(scenario_a, key=lambda entry: entry["pr_auc"], reverse=True)
    print(f"{stage}, skenario A, PR-AUC dari yang tertinggi: " + ", ".join(f"{entry['algorithm']} {entry['pr_auc']:.3f}" for entry in ranked))
print(f"Selisih terbesar antara PR-AUC data uji dan CV: {max(abs(entry['pr_auc'] - entry['cv_pr_auc']) for entry in metrics):.3f}.")
pd.DataFrame(metrics).query("scenario in ['A', 'B']").pivot(index=["stage", "algorithm"], columns="scenario", values="pr_auc").assign(
    selisih=lambda table: table["A"] - table["B"]
)

T1, skenario A, PR-AUC dari yang tertinggi: xgb 0.759, lr 0.744, rf 0.734
T2, skenario A, PR-AUC dari yang tertinggi: xgb 0.867, lr 0.852, rf 0.847
T3, skenario A, PR-AUC dari yang tertinggi: xgb 0.889, rf 0.888, lr 0.879
Selisih terbesar antara PR-AUC data uji dan CV: 0.051.


scenario             A      B  selisih
stage algorithm                       
T1    lr         0.744  0.569    0.175
      rf         0.734  0.589    0.145
      xgb        0.759  0.620    0.139
T2    lr         0.852  0.798    0.054
      rf         0.847  0.802    0.045
      xgb        0.867  0.833    0.034
T3    lr         0.879  0.846    0.034
      rf         0.888  0.855    0.033
      xgb        0.889  0.869    0.020

**Catatan penafsiran.** Urutan dan selisih di atas hanya nilai titik pada satu data uji. Selisih baru dapat disebut bermakna setelah interval kepercayaan bootstrap dihitung. Status target dan jawaban rumusan masalah ditulis setelah bootstrap dan SHAP selesai. Skenario D memakai baris uji tanpa `Enrolled`, dengan proporsi dropout yang lebih tinggi, jadi skornya tidak dapat dibandingkan langsung dengan skenario lain. Selisih akibat penghapusan atribut keuangan menunjukkan ketergantungan prediksi, bukan hubungan sebab akibat. Skor T3 tetap perlu dibaca dengan keterbatasan bahwa tahap ini mungkin terlalu optimistis. Hasil ini tidak mengubah parameter, ambang, atau atribut model.

### Bootstrap berpasangan

Satu matriks indeks 2.000 × 842 dipakai untuk semua perbandingan, jadi kedua model dalam satu perbandingan selalu dinilai pada baris yang sama. Interval kepercayaan memakai persentil 2,5 dan 97,5. Selisih disebut bermakna jika intervalnya tidak memuat 0 dan nilai absolutnya minimal 0,05. Skenario D tidak di-bootstrap karena baris ujinya berbeda, dan selisih ROC-AUC-nya hanya menjadi pembanding dengan literatur.

In [29]:
def percentile_ci(values):
    low, high = np.percentile(values, [2.5, 97.5])
    return round(float(low), 6), round(float(high), 6)


def is_meaningful(difference, ci_low, ci_high):
    return (ci_low > 0 or ci_high < 0) and abs(difference) >= MIN_DIFFERENCE


assert percentile_ci(np.arange(1001)) == (25.0, 975.0), percentile_ci(np.arange(1001))
assert is_meaningful(0.06, 0.01, 0.1) and is_meaningful(-0.06, -0.1, -0.01), "Selisih searah dan cukup besar harus bermakna"
assert is_meaningful(0.05, 0.01, 0.09), "Selisih tepat 0,05 harus bermakna"
assert not is_meaningful(0.04, 0.01, 0.07), "Selisih di bawah 0,05 tidak boleh bermakna"
assert not is_meaningful(0.06, -0.01, 0.1), "Interval yang memuat 0 tidak boleh bermakna"
assert not is_meaningful(0.06, 0.0, 0.1), "Interval yang tepat menyentuh 0 tidak boleh bermakna"

In [30]:
# Baris uji diurutkan menurut row_id, supaya artefak mudah dibaca dan tidak bergantung pada urutan pengacakan.
test_rows = test.sort_index()
y_test = test_rows["dropout"].to_numpy()
BOOTSTRAP = np.random.default_rng(SEED).integers(0, len(test_rows), size=(2000, len(test_rows)))
assert BOOTSTRAP.shape == (2000, 842), BOOTSTRAP.shape


def predict_risk(model_id, frame):
    model = searches[model_id].best_estimator_
    return model.predict_proba(frame[list(model.feature_names_in_)])[:, 1]


by_id = {entry["model_id"]: entry for entry in metrics}
boot_scores = {}
for model_id in SCENARIO_A + SCENARIO_B + list(C_REMOVED):
    risk = predict_risk(model_id, test_rows)
    assert round(float(average_precision_score(y_test, risk)), 6) == by_id[model_id]["pr_auc"], model_id
    boot_scores[model_id] = np.array([average_precision_score(y_test[rows], risk[rows]) for rows in BOOTSTRAP])


def compare(comparison_id, question, stage, left, right):
    difference = round(by_id[left]["pr_auc"] - by_id[right]["pr_auc"], 6)
    ci_low, ci_high = percentile_ci(boot_scores[left] - boot_scores[right])
    return {
        "id": comparison_id,
        "question": question,
        "metric": "pr_auc",
        "stage": stage,
        "left": left,
        "right": right,
        "difference": difference,
        "ci_low": ci_low,
        "ci_high": ci_high,
        "meaningful": is_meaningful(difference, ci_low, ci_high),
        "resamples": len(BOOTSTRAP),
    }


# NB-10
comparisons = []
for stage in WITH_FINANCIAL:
    for other in ["lr", "xgb"]:
        comparisons.append(compare(f"RM1_{stage}_rf-{other}", "RM1", stage, f"A_{stage}_rf", f"A_{stage}_{other}"))
for algorithm in ["rf", "lr", "xgb"]:
    for later, earlier in [("T2", "T1"), ("T3", "T2")]:
        comparisons.append(
            compare(f"RM1_{later}-{earlier}_{algorithm}", "RM1", f"{later}-{earlier}", f"A_{later}_{algorithm}", f"A_{earlier}_{algorithm}")
        )
for stage in WITH_FINANCIAL:
    for algorithm in ["rf", "lr", "xgb"]:
        comparisons.append(compare(f"RM2_{stage}_{algorithm}", "RM2", stage, f"A_{stage}_{algorithm}", f"B_{stage}_{algorithm}"))
for model_id in C_REMOVED:
    comparisons.append(compare(f"RM2_T1_rf_{model_id.rsplit('_', 1)[1]}", "RM2", "T1", "A_T1_rf", model_id))
for stage in WITH_FINANCIAL:
    left, right = f"D_{stage}_rf", f"A_{stage}_rf"
    comparisons.append(
        {
            "id": left,
            "question": "D",
            "metric": "roc_auc",
            "stage": stage,
            "left": left,
            "right": right,
            "difference": round(by_id[left]["roc_auc"] - by_id[right]["roc_auc"], 6),
            "ci_low": None,
            "ci_high": None,
            "meaningful": None,
            "resamples": 0,
        }
    )

questions = pd.Series([entry["question"] for entry in comparisons]).value_counts().to_dict()
assert questions == {"RM1": 12, "RM2": 12, "D": 3}, questions
assert len({entry["id"] for entry in comparisons}) == 27, "id perbandingan harus unik"
for entry in comparisons:
    if entry["question"] == "D":
        assert entry["ci_low"] is None and entry["ci_high"] is None and entry["meaningful"] is None, entry
    else:
        assert entry["ci_low"] <= entry["difference"] <= entry["ci_high"], entry

write_artifact("comparisons.json", comparisons)
pd.DataFrame(comparisons)[["id", "metric", "left", "right", "difference", "ci_low", "ci_high", "meaningful"]]

artifacts/comparisons.json ditulis, 7 KB.


,id,metric,left,right,difference,ci_low,ci_high,meaningful
0,RM1_T1_rf-lr,pr_auc,A_T1_rf,A_T1_lr,-9.202e-03,-0.036,1.734e-02,False
1,RM1_T1_rf-xgb,pr_auc,A_T1_rf,A_T1_xgb,-2.507e-02,-0.050,-7.000e-04,False
2,RM1_T2_rf-lr,pr_auc,A_T2_rf,A_T2_lr,-5.189e-03,-0.030,2.584e-02,False
3,RM1_T2_rf-xgb,pr_auc,A_T2_rf,A_T2_xgb,-2.020e-02,-0.035,-6.928e-03,False
4,RM1_T3_rf-lr,pr_auc,A_T3_rf,A_T3_lr,8.679e-03,-0.012,3.393e-02,False
5,RM1_T3_rf-xgb,pr_auc,A_T3_rf,A_T3_xgb,-7.080e-04,-0.011,1.047e-02,False
6,RM1_T2-T1_rf,pr_auc,A_T2_rf,A_T1_rf,1.128e-01,0.076,1.477e-01,True
7,RM1_T3-T2_rf,pr_auc,A_T3_rf,A_T2_rf,4.079e-02,0.022,6.130e-02,False
8,RM1_T2-T1_lr,pr_auc,A_T2_lr,A_T1_lr,1.088e-01,0.071,1.455e-01,True
9,RM1_T3-T2_lr,pr_auc,A_T3_lr,A_T2_lr,2.692e-02,0.010,4.355e-02,False


### SHAP

`TreeExplainer` dijalankan pada enam Random Forest skenario A dan B, untuk 842 baris uji. Nilai kolom one-hot dijumlahkan dahulu ke atribut asalnya per baris, baru kemudian dihitung rata-rata nilai absolutnya. Sel berikut menguji kedua langkah itu pada angka hitungan tangan.

In [31]:
def source_attribute(column):
    group, name = column.split("__", 1)
    return name if group == "remainder" else next(a for a in NOMINAL if name.startswith(f"{a}_"))


def aggregate_shap(columns, values, attributes):
    sources = np.array([source_attribute(column) for column in columns])
    return pd.DataFrame({attribute: values[:, sources == attribute].sum(axis=1) for attribute in attributes})


def top_attributes(attributes, values, k=5):
    order = np.argsort(-np.abs(values), kind="stable")[:k]
    return [[attributes[i], round(float(values[i]), 6)] for i in order]


toy = aggregate_shap(
    ["nominal__Course_33", "nominal__Course_infrequent_sklearn", "remainder__Debtor"],
    np.array([[0.5, -0.5, 0.25], [-0.25, 0.75, -0.25]]),
    ["Course", "Debtor"],
)
assert toy.to_dict("list") == {"Course": [0.0, 0.5], "Debtor": [0.25, -0.25]}, toy
assert toy.abs().mean().to_dict() == {"Course": 0.25, "Debtor": 0.25}, toy
assert source_attribute("nominal__Previous qualification_3") == "Previous qualification"
assert source_attribute("remainder__Previous qualification (grade)") == "Previous qualification (grade)"
toy_top = top_attributes(list("abcdef"), np.array([0.1, -0.4, 0.3, 0.0, -0.2, 0.05]))
assert toy_top == [["b", -0.4], ["c", 0.3], ["e", -0.2], ["a", 0.1], ["f", 0.05]], toy_top

In [32]:
SHAP_MODELS = [f"{scenario}_{stage}_rf" for scenario in "AB" for stage in WITH_FINANCIAL]


def explain(model, frame):
    encoded = model["enc"].transform(frame[list(model.feature_names_in_)])
    explainer = shap.TreeExplainer(model["clf"])
    return explainer.shap_values(encoded)[:, :, 1], float(explainer.expected_value[1])


started = time.perf_counter()
explained = joblib.Parallel(n_jobs=-1)(
    joblib.delayed(explain)(searches[model_id].best_estimator_, test_rows) for model_id in SHAP_MODELS
)
timings["shap"] += time.perf_counter() - started

shap_by_attribute = {}
shap_global = []
test_predictions = []
for model_id, (values, base_value) in zip(SHAP_MODELS, explained):
    model = searches[model_id].best_estimator_
    risk = predict_risk(model_id, test_rows)
    # NB-11: diperiksa pada seluruh baris uji, lebih ketat daripada 20 baris contoh.
    gap = np.abs(base_value + values.sum(axis=1) - risk).max()
    assert gap <= 1e-6, f"{model_id}: base_value + jumlah SHAP berbeda dari risk sebesar {gap}"

    attributes = list(model.feature_names_in_)
    by_attribute = aggregate_shap(model["enc"].get_feature_names_out(), values, attributes)
    assert np.allclose(by_attribute.sum(axis=1), values.sum(axis=1)), f"{model_id}: agregasi mengubah jumlah SHAP"
    shap_by_attribute[model_id] = by_attribute

    importance = by_attribute.abs().mean().sort_values(ascending=False, kind="stable")
    shap_global += [
        {"model_id": model_id, "attribute": attribute, "mean_abs_shap": round(float(value), 6), "rank": rank}
        for rank, (attribute, value) in enumerate(importance.items(), start=1)
    ]

    rows = by_attribute.to_numpy()
    for position, (row_id, status) in enumerate(zip(test_rows.index, test_rows["Target"])):
        test_predictions.append(
            {
                "row_id": int(row_id),
                "model_id": model_id,
                "risk": round(float(risk[position]), 6),
                "flagged": bool(risk[position] >= THRESHOLD),
                "status": status,
                "top": top_attributes(attributes, rows[position]),
            }
        )

assert len(shap_global) == 171, len(shap_global)
assert len(test_predictions) == 5052, len(test_predictions)

# Metrik ambang yang dihitung ulang dari test_predictions harus sama dengan metrics.json.
for model_id in SHAP_MODELS:
    entries = [entry for entry in test_predictions if entry["model_id"] == model_id]
    tp = sum(entry["flagged"] and entry["status"] == "Dropout" for entry in entries)
    fp = sum(entry["flagged"] and entry["status"] != "Dropout" for entry in entries)
    fn = sum(not entry["flagged"] and entry["status"] == "Dropout" for entry in entries)
    assert (tp, fp, fn) == (by_id[model_id]["tp"], by_id[model_id]["fp"], by_id[model_id]["fn"]), model_id

write_artifact("shap_global.json", shap_global)
write_artifact("test_predictions.json", test_predictions, indent=None)

/usr/local/lib/python3.13/dist-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


artifacts/shap_global.json ditulis, 21 KB.
artifacts/test_predictions.json ditulis, 1476 KB.


### Jawaban rumusan masalah

Seluruh angka di bawah dihitung dari hasil jalan ini. Interval pada rumusan masalah 1 dan 2 adalah interval selisih PR-AUC, dan interval pada rumusan masalah 3 adalah interval rata-rata nilai absolut SHAP, keduanya dari matriks bootstrap yang sama.

In [33]:
def comparison_table(entries):
    return pd.DataFrame(entries)[["id", "difference", "ci_low", "ci_high", "meaningful"]]


def meaningful_summary(entries):
    hits = [entry["id"] for entry in entries if entry["meaningful"]]
    return f"{len(hits)} dari {len(entries)} selisih bermakna" + (f" ({', '.join(hits)})" if hits else "")


algorithm_gaps = [entry for entry in comparisons if entry["question"] == "RM1" and "-" not in entry["stage"]]
stage_gaps = [entry for entry in comparisons if entry["question"] == "RM1" and "-" in entry["stage"]]

print("RM1. Random Forest pada skenario A:")
for stage in WITH_FINANCIAL:
    entry = by_id[f"A_{stage}_rf"]
    highest = max((by_id[f"A_{stage}_{algorithm}"] for algorithm in ["rf", "lr", "xgb"]), key=lambda item: item["pr_auc"])
    print(
        f"  {stage}: PR-AUC {entry['pr_auc']:.3f}, recall {entry['recall']:.3f}, presisi {entry['precision']:.3f}, "
        f"F1 {entry['f1']:.3f}, ROC-AUC {entry['roc_auc']:.3f}. "
        f"PR-AUC titik tertinggi pada tahap ini: {highest['algorithm']} {highest['pr_auc']:.3f}."
    )
print(f"Random Forest melawan pembanding: {meaningful_summary(algorithm_gaps)}.")
print(f"Antartahap: {meaningful_summary(stage_gaps)}.")
display(comparison_table(algorithm_gaps), comparison_table(stage_gaps))

RM1. Random Forest pada skenario A:
  T1: PR-AUC 0.734, recall 0.638, presisi 0.679, F1 0.658, ROC-AUC 0.826. PR-AUC titik tertinggi pada tahap ini: xgb 0.759.
  T2: PR-AUC 0.847, recall 0.705, presisi 0.815, F1 0.756, ROC-AUC 0.896. PR-AUC titik tertinggi pada tahap ini: xgb 0.867.
  T3: PR-AUC 0.888, recall 0.772, presisi 0.838, F1 0.804, ROC-AUC 0.927. PR-AUC titik tertinggi pada tahap ini: xgb 0.889.
Random Forest melawan pembanding: 0 dari 6 selisih bermakna.
Antartahap: 3 dari 6 selisih bermakna (RM1_T2-T1_rf, RM1_T2-T1_lr, RM1_T2-T1_xgb).


,id,difference,ci_low,ci_high,meaningful
0,RM1_T1_rf-lr,-9.202e-03,-0.036,1.734e-02,False
1,RM1_T1_rf-xgb,-2.507e-02,-0.050,-7.000e-04,False
2,RM1_T2_rf-lr,-5.189e-03,-0.030,2.584e-02,False
3,RM1_T2_rf-xgb,-2.020e-02,-0.035,-6.928e-03,False
4,RM1_T3_rf-lr,8.679e-03,-0.012,3.393e-02,False
5,RM1_T3_rf-xgb,-7.080e-04,-0.011,1.047e-02,False


,id,difference,ci_low,ci_high,meaningful
0,RM1_T2-T1_rf,0.113,0.076,0.148,True
1,RM1_T3-T2_rf,0.041,0.022,0.061,False
2,RM1_T2-T1_lr,0.109,0.071,0.146,True
3,RM1_T3-T2_lr,0.027,0.010,0.044,False
4,RM1_T2-T1_xgb,0.108,0.076,0.142,True
5,RM1_T3-T2_xgb,0.021,0.004,0.039,False


In [34]:
financial_gaps = [entry for entry in comparisons if entry["question"] == "RM2" and entry["id"].count("_") == 2]
single_gaps = [entry for entry in comparisons if entry["question"] == "RM2" and entry["id"].count("_") == 3]

print(f"RM2. Skenario A dikurangi B: {meaningful_summary(financial_gaps)}.")
print(f"A_T1_rf dikurangi model tanpa satu atribut keuangan: {meaningful_summary(single_gaps)}.")
display(comparison_table(financial_gaps), comparison_table(single_gaps))

print("Pembanding literatur, selisih ROC-AUC skenario D dikurangi A pada Random Forest, tanpa bootstrap:")
pd.DataFrame([entry for entry in comparisons if entry["question"] == "D"])[["id", "left", "right", "difference"]]

RM2. Skenario A dikurangi B: 4 dari 9 selisih bermakna (RM2_T1_rf, RM2_T1_lr, RM2_T1_xgb, RM2_T2_lr).
A_T1_rf dikurangi model tanpa satu atribut keuangan: 1 dari 3 selisih bermakna (RM2_T1_rf_no-tuition).


,id,difference,ci_low,ci_high,meaningful
0,RM2_T1_rf,0.145,0.101,0.184,True
1,RM2_T1_lr,0.175,0.130,0.217,True
2,RM2_T1_xgb,0.139,0.094,0.180,True
3,RM2_T2_rf,0.045,0.024,0.067,False
4,RM2_T2_lr,0.054,0.031,0.075,True
5,RM2_T2_xgb,0.034,0.013,0.055,False
6,RM2_T3_rf,0.033,0.020,0.047,False
7,RM2_T3_lr,0.034,0.013,0.051,False
8,RM2_T3_xgb,0.020,0.002,0.036,False


,id,difference,ci_low,ci_high,meaningful
0,RM2_T1_rf_no-debtor,-0.006,-0.018,0.006,False
1,RM2_T1_rf_no-tuition,0.113,0.077,0.146,True
2,RM2_T1_rf_no-scholarship,0.005,-0.010,0.020,False


Pembanding literatur, selisih ROC-AUC skenario D dikurangi A pada Random Forest, tanpa bootstrap:


,id,left,right,difference
0,D_T1_rf,D_T1_rf,A_T1_rf,0.028
1,D_T2_rf,D_T2_rf,A_T2_rf,0.037
2,D_T3_rf,D_T3_rf,A_T3_rf,0.028


In [35]:
def top_five(model_id):
    rows = []
    for entry in shap_global:
        if entry["model_id"] == model_id and entry["rank"] <= 5:
            magnitudes = shap_by_attribute[model_id][entry["attribute"]].abs().to_numpy()
            ci_low, ci_high = percentile_ci(magnitudes[BOOTSTRAP].mean(axis=1))
            rows.append(
                {
                    "rank": entry["rank"],
                    "atribut": entry["attribute"],
                    "mean_abs_shap": entry["mean_abs_shap"],
                    "ci_low": ci_low,
                    "ci_high": ci_high,
                }
            )
    return pd.DataFrame(rows).set_index("rank")


print("RM3. Lima atribut teratas per tahap, dengan dan tanpa atribut keuangan:")
for stage in WITH_FINANCIAL:
    with_financial, without_financial = top_five(f"A_{stage}_rf"), top_five(f"B_{stage}_rf")
    entered = [attribute for attribute in without_financial["atribut"] if attribute not in set(with_financial["atribut"])]
    print(f"{stage}: masuk lima besar setelah atribut keuangan dihapus: {entered}")
    display(pd.concat({"Dengan keuangan": with_financial, "Tanpa keuangan": without_financial}, axis=1))

RM3. Lima atribut teratas per tahap, dengan dan tanpa atribut keuangan:
T1: masuk lima besar setelah atribut keuangan dihapus: ['Gender', 'Previous qualification']


Dengan keuangan                               \
                      atribut mean_abs_shap ci_low ci_high   
rank                                                         
1     Tuition fees up to date         0.079  0.074   0.084   
2           Age at enrollment         0.052  0.051   0.054   
3          Scholarship holder         0.048  0.046   0.050   
4            Application mode         0.040  0.039   0.042   
5                      Course         0.039  0.037   0.041   

              Tanpa keuangan                               
                     atribut mean_abs_shap ci_low ci_high  
rank                                                       
1          Age at enrollment         0.055  0.053   0.056  
2           Application mode         0.047  0.045   0.049  
3                     Course         0.045  0.042   0.047  
4                     Gender         0.037  0.036   0.038  
5     Previous qualification         0.017  0.016   0.018

T2: masuk lima besar setelah atribut keuangan dihapus: ['Course', 'Application mode']


Dengan keuangan                               \
                                  atribut mean_abs_shap ci_low ci_high   
rank                                                                     
1     Curricular units 1st sem (approved)         0.092  0.090   0.095   
2        Curricular units 1st sem (grade)         0.061  0.058   0.063   
3                 Tuition fees up to date         0.048  0.045   0.052   
4                       Age at enrollment         0.033  0.032   0.033   
5                      Scholarship holder         0.027  0.026   0.028   

                           Tanpa keuangan                               
                                  atribut mean_abs_shap ci_low ci_high  
rank                                                                    
1     Curricular units 1st sem (approved)         0.122  0.119   0.126  
2        Curricular units 1st sem (grade)         0.067  0.064   0.070  
3                       Age at enrollment         0.043  0.042   0.044  
4                                  Course         0.028  0.027   0.030  
5                        Application mode         0.028  0.026   0.029

T3: masuk lima besar setelah atribut keuangan dihapus: ['Age at enrollment']


Dengan keuangan                               \
                                  atribut mean_abs_shap ci_low ci_high   
rank                                                                     
1     Curricular units 2nd sem (approved)         0.099  0.097   0.102   
2        Curricular units 2nd sem (grade)         0.064  0.062   0.066   
3     Curricular units 1st sem (approved)         0.060  0.059   0.062   
4                 Tuition fees up to date         0.048  0.044   0.051   
5        Curricular units 1st sem (grade)         0.027  0.026   0.028   

                           Tanpa keuangan                               
                                  atribut mean_abs_shap ci_low ci_high  
rank                                                                    
1     Curricular units 2nd sem (approved)         0.092  0.090   0.095  
2     Curricular units 1st sem (approved)         0.061  0.060   0.063  
3        Curricular units 2nd sem (grade)         0.060  0.058   0.062  
4        Curricular units 1st sem (grade)         0.029  0.028   0.031  
5                       Age at enrollment         0.028  0.027   0.029

### Status target

Proposal Subbab 5.4 tidak menyebut skenario untuk target Random Forest, dan hal ini baru disadari setelah hasil data uji terlihat. Status G2 ditetapkan memakai skenario A, dan penetapan ini dilakukan setelah hasil terlihat, jadi perlu dibaca dengan hati-hati. Skenario B tetap ditampilkan sebagai pembanding. Target dan batas selisih 0,05 adalah patokan awal, bukan standar baku.

In [36]:
def target_status(scenario):
    t1, t2 = by_id[f"{scenario}_T1_rf"], by_id[f"{scenario}_T2_rf"]
    t1_met = t1["pr_auc"] >= 0.60
    t2_met = t2["pr_auc"] >= 0.80 and t2["recall"] >= 0.70
    return {
        "skenario": scenario,
        "T1_pr_auc": t1["pr_auc"],
        "T2_pr_auc": t2["pr_auc"],
        "T2_recall": t2["recall"],
        "target_T1": t1_met,
        "target_T2": t2_met,
        "G2": t1_met and t2_met,
        "tahap_paling_dini": "T1" if t1_met else "T2" if t2_met else "tidak ada",
    }


official = target_status("A")
print(
    f"Status G2 memakai skenario A: {'tercapai' if official['G2'] else 'tidak tercapai'}. "
    f"Tahap paling dini yang memenuhi target: {official['tahap_paling_dini']}. Skenario B hanya pembanding."
)
pd.DataFrame([official, target_status("B")])

Status G2 memakai skenario A: tercapai. Tahap paling dini yang memenuhi target: T1. Skenario B hanya pembanding.


,skenario,T1_pr_auc,T2_pr_auc,T2_recall,target_T1,target_T2,G2,tahap_paling_dini
0,A,0.734,0.847,0.705,True,True,True,T1
1,B,0.589,0.802,0.627,False,False,False,tidak ada


**Penafsiran.**

- **Rumusan masalah 1.** Random Forest tidak berbeda bermakna dari Regresi Logistik dan XGBoost pada tahap mana pun, jadi pilihan algoritma bukan pembeda utama pada data ini. Kenaikan terbesar terjadi dari T1 ke T2. Kenaikan dari T2 ke T3 berada di bawah batas 0,05, sehingga akhir semester 1 sudah memberi sebagian besar manfaat.
- **Rumusan masalah 2.** Ketergantungan pada atribut keuangan paling besar pada T1 dan mengecil setelah nilai semester tersedia. Pada T2, hanya Regresi Logistik yang masih melewati batas, dan pada T3 tidak ada. Skenario C menunjukkan bahwa ketergantungan pada T1 terutama berasal dari `Tuition fees up to date`. Waktu pencatatan atribut ini tidak dirinci pada sumber data, jadi skor T1 yang memakainya bisa saja terlalu optimistis, dan skenario B memberi batas bawahnya.
- **Rumusan masalah 3.** `Tuition fees up to date` tetap masuk lima besar pada ketiga tahap, dan atribut semester memimpin setelah tersedia. Saat atribut keuangan dihapus, atribut demografi dan akademik saat masuk naik peringkat.

Beberapa interval tidak memuat 0 tetapi selisihnya di bawah 0,05, misalnya Random Forest melawan XGBoost pada T2. Selisih seperti ini tidak disebut bermakna menurut aturan rancangan, walaupun arahnya terdeteksi. Interval menggambarkan ketidakpastian data uji, bukan variasi antarpembagian data. Hasil ini menunjukkan pola prediktif, bukan hubungan sebab akibat.

## 7. Pemanfaatan: ekspor artefak

Bagian ini menulis sisa artefak untuk aplikasi Streamlit, menyimpan enam Random Forest, dan membuat `artifacts.zip`, yang isinya langsung berupa file artefak tanpa folder induk. Di Kaggle, `artifacts.zip` dan notebook yang sudah dijalankan diunduh dari tab Output.

In [37]:
ATTRIBUTES = [column for column in raw.columns if column != "Target"]
test_rows[ATTRIBUTES + ["Target"]].to_csv(ARTIFACTS / "test_rows.csv", index_label="row_id")


def plain(attribute, value):
    return int(value) if pd.api.types.is_integer_dtype(df[attribute]) and float(value).is_integer() else float(value)


def form_attributes(model_id):
    available = list(searches[model_id].best_estimator_.feature_names_in_)
    chosen = [attribute for attribute in FINANCIAL if attribute in available]
    for semester in ["1st", "2nd"]:
        fields = [f"Curricular units {semester} sem ({field})" for field in ["enrolled", "evaluations", "approved", "grade"]]
        chosen += [attribute for attribute in fields if attribute in available]
    ranked = [entry["attribute"] for entry in shap_global if entry["model_id"] == model_id]
    return chosen + [attribute for attribute in ranked if attribute not in chosen][: 15 - len(chosen)]


form_spec = {
    "defaults": {
        **{attribute: plain(attribute, train[attribute].median()) for attribute in NUMERIC},
        **{attribute: int(train[attribute].mode().iloc[0]) for attribute in NOMINAL + BINARY},
    },
    "ranges": {attribute: [plain(attribute, df[attribute].min()), plain(attribute, df[attribute].max())] for attribute in NUMERIC},
    "categories": {attribute: sorted(int(value) for value in df[attribute].unique()) for attribute in NOMINAL + BINARY},
    "grade_nonzero_range": {
        semester: [float(grades[grades > 0].min()), float(grades.max())]
        for semester, grades in (("1st", df[SEM1[4]]), ("2nd", df[SEM2[4]]))
    },
    "models": {model_id: form_attributes(model_id) for model_id in SHAP_MODELS},
}

assert set(form_spec["defaults"]) == set(ATTRIBUTES), "defaults harus memuat 36 atribut"
for model_id, attributes in form_spec["models"].items():
    available = set(searches[model_id].best_estimator_.feature_names_in_)
    assert len(attributes) == 15 and len(set(attributes)) == 15 and set(attributes) <= available, f"{model_id}: {attributes}"
    assert (set(FINANCIAL) & available) <= set(attributes), f"{model_id}: atribut keuangan yang dipakai model harus tampil"

# Nilai defaults harus lolos aturan V1 sampai V4 pada form.
for semester in ["1st", "2nd"]:
    value = lambda field: form_spec["defaults"][f"Curricular units {semester} sem ({field})"]
    low, high = form_spec["grade_nonzero_range"][semester]
    assert value("approved") <= value("enrolled"), f"{semester}: V1"
    assert value("evaluations") >= value("approved"), f"{semester}: V2"
    assert (value("approved") == 0) == (value("grade") == 0), f"{semester}: V3"
    assert value("grade") == 0 or low <= value("grade") <= high, f"{semester}: V4"

write_artifact("form_spec.json", form_spec)

artifacts/form_spec.json ditulis, 8 KB.


In [38]:
# NB-14: n_jobs diatur ke 1 supaya prediksi satu baris tidak memakan biaya paralelisme.
(ARTIFACTS / "models").mkdir(parents=True, exist_ok=True)
for model_id in SHAP_MODELS:
    pipeline = searches[model_id].best_estimator_
    pipeline["clf"].n_jobs = 1
    joblib.dump(pipeline, ARTIFACTS / "models" / f"{model_id}.joblib", compress=3)

manifest = {
    "schema_version": 1,
    "created_at": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "run_env": run_env,
    "seed": SEED,
    "threshold": THRESHOLD,
    "data_sha256": data_sha256,
    "versions": {
        "python": platform.python_version(),
        **{package: version(package) for package in ["numpy", "pandas", "scikit-learn", "xgboost", "shap"]},
    },
    "timings_seconds": {
        "tuning": round(timings["tuning"], 1),
        "shap": round(timings["shap"], 1),
        "total": round(time.perf_counter() - START, 1),
    },
}
write_artifact("manifest.json", manifest)

ARTIFACT_FILES = [
    "manifest.json",
    "dataset_summary.json",
    "metrics.json",
    "comparisons.json",
    "shap_global.json",
    "test_predictions.json",
    "test_rows.csv",
    "form_spec.json",
    *[f"models/{model_id}.joblib" for model_id in SHAP_MODELS],
]
with zipfile.ZipFile(ROOT / "artifacts.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    for name in ARTIFACT_FILES:
        archive.write(ARTIFACTS / name, arcname=name)

artifacts/manifest.json ditulis, 0 KB.


In [39]:
# Pemeriksaan akhir dari file di disk, bukan dari variabel di memori.
def read_artifact(name):
    return json.loads((ARTIFACTS / name).read_text(encoding="utf-8"))


with zipfile.ZipFile(ROOT / "artifacts.zip") as archive:
    assert archive.namelist() == ARTIFACT_FILES, archive.namelist()

stored_counts = {name: len(read_artifact(name)) for name in ["metrics.json", "comparisons.json", "shap_global.json", "test_predictions.json"]}
assert stored_counts == {"metrics.json": 24, "comparisons.json": 27, "shap_global.json": 171, "test_predictions.json": 5052}, stored_counts
assert read_artifact("manifest.json")["schema_version"] == 1

stored_rows = pd.read_csv(ARTIFACTS / "test_rows.csv", index_col="row_id")
assert stored_rows.shape == (842, 37), stored_rows.shape
assert stored_rows.equals(test_rows[ATTRIBUTES + ["Target"]]), "test_rows.csv tidak sama dengan data uji setelah dibaca ulang"

stored_predictions = read_artifact("test_predictions.json")
sample = stored_rows.index[:20]
for model_id in SHAP_MODELS:
    loaded = joblib.load(ARTIFACTS / "models" / f"{model_id}.joblib")
    assert loaded["clf"].n_jobs == 1, model_id
    expected = {entry["row_id"]: entry["risk"] for entry in stored_predictions if entry["model_id"] == model_id}
    scores = loaded.predict_proba(stored_rows.loc[sample, list(loaded.feature_names_in_)])[:, 1]
    assert [round(float(score), 6) for score in scores] == [expected[row_id] for row_id in sample], f"{model_id}: skor model yang dimuat ulang berbeda"

archive_size = (ROOT / "artifacts.zip").stat().st_size / 1024**2
print(f"Seluruh artefak lolos pemeriksaan. artifacts.zip: {len(ARTIFACT_FILES)} file, {archive_size:.1f} MB.")
print("Durasi dalam detik:", manifest["timings_seconds"])

Seluruh artefak lolos pemeriksaan. artifacts.zip: 14 file, 47.1 MB.
Durasi dalam detik: {'tuning': 1130.0, 'shap': 905.5, 'total': 2109.1}


## Keterbatasan

- **Satu institusi.** Data berasal dari satu politeknik di Portugal, sehingga hasilnya belum tentu berlaku di Indonesia.
- **Tanpa penanda waktu.** Waktu pencatatan atribut keuangan tidak dirinci. Jika sebagian dicatat setelah kuliah berjalan, skor T1 yang memakainya terlalu optimistis. Skenario B memberi batas bawahnya.
- **T3 mungkin terlalu optimistis.** Martins dkk. (2023) membuang mahasiswa yang sudah keluar pada tahap ini, sedangkan file publik tidak memiliki tanggal keluar. Karena itu, fokus utama ada pada T1 dan T2.
- **Definisi target.** Mahasiswa `Enrolled` masih mungkin keluar setelah masa studi normal.
- **Program yang dikeluarkan.** Hasil tidak mencakup `Course` 171.
- **Satu pembagian data.** Interval kepercayaan bootstrap menggambarkan ketidakpastian data uji, tetapi bukan variasi antarpembagian data.
- **Target sebagai patokan awal.** Target Random Forest dan batas selisih 0,05 bukan standar baku. Skenario untuk status G2 ditetapkan setelah hasil data uji terlihat.
- **Pola prediktif.** Hasil menunjukkan pola prediktif, bukan hubungan sebab akibat. Model memakai bobot kelas, jadi skornya bukan peluang terkalibrasi.

Durasi yang dicetak di atas berasal dari jalan notebook ini. `manifest.json` pada Release `artifacts-v1` mencatat durasi jalan yang menghasilkan artefak final, jadi angkanya dapat berbeda walaupun hasilnya identik.